# LexData — Modelo Predictivo · Nicho Familiar · v1

Pipeline completo: **carga desde PostgreSQL → ingeniería de features → comparación de modelos
→ XGBoost con log1p → survival analysis (Cox PH) → alertas tempranas → serialización**.

| Sección | Contenido |
|---|---|
| 1 | Setup y dependencias |
| 2 | Carga de datos desde la DB |
| 3 | Generación de expedientes sintéticos |
| 4 | Ingeniería de features y split temporal |
| 5 | Comparación de modelos (Ridge / Lasso / GBM / XGBoost) |
| 6 | Entrenamiento final XGBoost con grid-search |
| 7 | Evaluación y diagnóstico de overfitting |
| 8 | Importancia de variables + SHAP |
| 9 | Survival Analysis — Cox PH + Kaplan-Meier |
| 10 | Alertas tempranas |
| 11 | Serialización y resumen de outputs |


## Sección 1 — Setup y dependencias

In [1]:
# !pip install xgboost lifelines shap scikit-learn sqlalchemy psycopg2-binary python-dotenv joblib

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

try:
    import xgboost as xgb
    XGB_OK = True
except ImportError:
    XGB_OK = False
    print("⚠ XGBoost no disponible — instalar con: pip install xgboost")

try:
    from lifelines import CoxPHFitter, KaplanMeierFitter
    from lifelines.statistics import logrank_test
    LIFELINES_OK = True
except ImportError:
    LIFELINES_OK = False
    print("⚠ lifelines no disponible — instalar con: pip install lifelines")

try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False
    print("⚠ SHAP no disponible — se usará importancia por permutación")

warnings.filterwarnings("ignore")
np.random.seed(42)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

print("✅ Setup completo")
print(f"   XGBoost:  {'✅' if XGB_OK else '⚠ no disponible'}")
print(f"   lifelines:{'✅' if LIFELINES_OK else '⚠ no disponible'}")
print(f"   SHAP:     {'✅' if SHAP_OK else '⚠ no disponible'}")


⚠ lifelines no disponible — instalar con: pip install lifelines
⚠ SHAP no disponible — se usará importancia por permutación
✅ Setup completo
   XGBoost:  ✅
   lifelines:⚠ no disponible
   SHAP:     ⚠ no disponible


## Sección 2 — Carga de datos desde PostgreSQL

In [2]:
# ── Conexión via .env ────────────────────────────────────────────────────────
load_dotenv()
DB_URL = (
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL)

def sql(query):
    with engine.connect() as con:
        return pd.read_sql(text(query), con)

print("🔌 Conectado a:", DB_URL.split("@")[-1])


🔌 Conectado a: localhost:5432/lexdata


In [3]:
# ── Cargar las 5 tablas ───────────────────────────────────────────────────────
df_ivf       = sql("SELECT * FROM lexdata.ivf_municipios")
df_vif       = sql("SELECT municipio, departamento, anio, SUM(cantidad) AS cantidad FROM lexdata.hechos_vif GROUP BY 1,2,3")
df_alimentos = sql("SELECT municipio, departamento, anio, SUM(cantidad) AS cantidad FROM lexdata.procesos_alimentos GROUP BY 1,2,3")
df_icbf      = sql("SELECT municipio, departamento, anio, SUM(cantidad) AS cantidad FROM lexdata.medidas_icbf GROUP BY 1,2,3")
df_inasist   = sql("SELECT municipio, departamento, anio, SUM(cantidad) AS cantidad FROM lexdata.inasistencia_alimentaria GROUP BY 1,2,3")

print(f"ivf_municipios  : {df_ivf.shape[0]:>6,} filas · {df_ivf['municipio'].nunique()} municipios")
print(f"hechos_vif      : {df_vif.shape[0]:>6,} filas")
print(f"procesos_alim   : {df_alimentos.shape[0]:>6,} filas")
print(f"medidas_icbf    : {df_icbf.shape[0]:>6,} filas")
print(f"inasistencia    : {df_inasist.shape[0]:>6,} filas")


ivf_municipios  :  1,061 filas · 1061 municipios
hechos_vif      :  6,637 filas
procesos_alim   :      6 filas
medidas_icbf    :     56 filas
inasistencia    :     56 filas


In [5]:
# ── limpieza base ─────────────────────────────────────────────
df_ivf["municipio"] = df_ivf["municipio"].astype(str).str.strip().str.upper()
df_ivf["anio"] = pd.to_numeric(df_ivf["anio"], errors="coerce")

df_ivf = df_ivf.dropna(subset=["municipio", "anio"])

# ── último año por municipio ───────────────────────────────────
ultimo_anio = (
    df_ivf.groupby("municipio")["anio"]
    .max()
    .reset_index()
    .rename(columns={"anio": "anio_max"})
)

# ── merge seguro ───────────────────────────────────────────────
df_ivf_base = df_ivf.merge(ultimo_anio, on="municipio")

df_ivf_base = df_ivf_base[df_ivf_base["anio"] == df_ivf_base["anio_max"]]

# ── seguridad final ────────────────────────────────────────────
df_ivf_base = df_ivf_base.dropna(subset=["ivf_score_ponderado"])

ALL_MUNICIPIOS = df_ivf_base["municipio"].tolist()

IVF_SCORES = dict(zip(
    df_ivf_base["municipio"],
    df_ivf_base["ivf_score_ponderado"]
))

print(f"Municipios únicos en IVF base : {len(ALL_MUNICIPIOS)}")

if len(IVF_SCORES) == 0:
    print("IVF_SCORES vacío: revisa df_ivf_base")
else:
    print(
        f"IVF score range: "
        f"{min(IVF_SCORES.values()):.2f} – {max(IVF_SCORES.values()):.2f}"
    )

print(f"Municipios en alerta (P75)    : {df_ivf_base['alerta'].sum()}")

Municipios únicos en IVF base : 0
IVF_SCORES vacío: revisa df_ivf_base
Municipios en alerta (P75)    : 0


## Sección 3 — Generación de expedientes sintéticos

> Los datos del CSJ (`procesos_alimentos`) proveen **volumen real** de procesos por municipio.
> Aquí usamos esos conteos reales como pesos de muestreo para generar registros de expedientes
> con duración estimada — estructura que requiere el modelo supervisado.
> **Cuando el CSJ exporte duraciones reales de SICOF, reemplazar `df_exp` directamente.**


In [6]:
TIPOS      = ["ALIMENTOS", "VIF", "HURTO_PATRIMONIAL", "SUSTANCIAS"]
PROBS_TIPO = [0.35, 0.35, 0.20, 0.10]

DESPACHOS  = [
    "Juzgado_1_Familia", "Juzgado_2_Familia", "Juzgado_3_Familia",
    "Comisaria_1_Familia", "Comisaria_2_Familia",
    "Juzgado_Penal_Municipal_1", "Juzgado_Penal_Municipal_2",
]
CARGA_DESPACHO = {
    "Juzgado_1_Familia": 1.35, "Juzgado_2_Familia": 1.20, "Juzgado_3_Familia": 0.95,
    "Comisaria_1_Familia": 1.10, "Comisaria_2_Familia": 0.90,
    "Juzgado_Penal_Municipal_1": 1.25, "Juzgado_Penal_Municipal_2": 1.05,
}
DUR_BASE = {"ALIMENTOS": 240, "VIF": 180, "HURTO_PATRIMONIAL": 280, "SUSTANCIAS": 200}
DUR_STD  = {"ALIMENTOS": 90,  "VIF": 70,  "HURTO_PATRIMONIAL": 110, "SUSTANCIAS": 80}
YEARS      = [2020, 2021, 2022, 2023, 2024]
YEAR_PROBS = [0.10, 0.15, 0.20, 0.25, 0.30]
N_EXP      = 12_000

# Pesos de muestreo = sqrt(ivf_score) para que todos los municipios tengan cobertura
ivf_arr = df_ivf_base["ivf_score_ponderado"].values
weights = np.sqrt(np.maximum(ivf_arr, 0.01))
weights /= weights.sum()

registros = []
for i in range(N_EXP):
    mun      = ALL_MUNICIPIOS[np.random.choice(len(ALL_MUNICIPIOS), p=weights)]
    tipo     = np.random.choice(TIPOS, p=PROBS_TIPO)
    despacho = np.random.choice(DESPACHOS)
    anio     = np.random.choice(YEARS, p=YEAR_PROBS)
    ivf_val  = IVF_SCORES.get(mun, 0.0)
    carga    = CARGA_DESPACHO[despacho]
    n_aud    = max(1, int(np.random.gamma(2, 2.5)))
    apel     = 1 if (tipo in ["ALIMENTOS","HURTO_PATRIMONIAL"] and np.random.random() > 0.65) else 0
    dur = max(30, int(
        DUR_BASE[tipo] * carga
        + ivf_val * 3.0
        + n_aud * 8
        + apel * 140
        + (2024 - anio) * (-3)
        + np.random.normal(0, DUR_STD[tipo] * 0.25)
    ))
    terminado = 1 if (dur <= 730 and np.random.random() > 0.15) else 0
    if not terminado:
        dur = int(dur * np.random.uniform(0.4, 0.9))
    registros.append({
        "expediente_id": f"EXP-{i+1:05d}", "municipio": mun,
        "tipo_proceso": tipo, "despacho": despacho, "anio_radicacion": anio,
        "ivf_score": ivf_val, "carga_despacho": carga,
        "n_audiencias": n_aud, "apelacion": apel,
        "duracion_dias": dur, "terminado": terminado,
    })

df_exp = pd.DataFrame(registros)

# Garantizar cobertura total de municipios
mun_faltantes = set(ALL_MUNICIPIOS) - set(df_exp["municipio"].unique())
if mun_faltantes:
    extra = [
        {"expediente_id": f"EXP-F-{j:05d}", "municipio": mun,
         "tipo_proceso": "ALIMENTOS", "despacho": "Juzgado_1_Familia",
         "anio_radicacion": 2023, "ivf_score": IVF_SCORES.get(mun, 0.0),
         "carga_despacho": 1.35, "n_audiencias": 4, "apelacion": 0,
         "duracion_dias": 264, "terminado": 1}
        for j, mun in enumerate(mun_faltantes) for _ in range(8)
    ]
    df_exp = pd.concat([df_exp, pd.DataFrame(extra)], ignore_index=True)
    print(f"  → {len(extra)} registros de cobertura añadidos")

assert not (set(ALL_MUNICIPIOS) - set(df_exp["municipio"].unique())), "Municipios faltantes!"

print(f"✅ Expedientes generados: {len(df_exp):,}")
print(f"   Duración media  : {df_exp['duracion_dias'].mean():.0f} días")
print(f"   Terminados      : {df_exp['terminado'].mean()*100:.1f} %")
print(f"   Municipios cubiertos: {df_exp['municipio'].nunique()}")


ValueError: a must be greater than 0 unless no samples are taken

## Sección 4 — Ingeniería de features y split temporal

In [ ]:
FEATURES = [
    "ivf_score", "carga_despacho", "n_audiencias", "apelacion",
    "anio_radicacion", "tipo_proceso_enc", "municipio_enc", "despacho_enc",
]

df_model = df_exp.copy()

le_tipo = LabelEncoder()
le_mun  = LabelEncoder()
le_desp = LabelEncoder()

df_model["tipo_proceso_enc"] = le_tipo.fit_transform(df_model["tipo_proceso"])
df_model["municipio_enc"]    = le_mun.fit_transform(df_model["municipio"])
df_model["despacho_enc"]     = le_desp.fit_transform(df_model["despacho"])

# Asegurar que todos los municipios del IVF estén en el LabelEncoder
mun_falt_enc = set(ALL_MUNICIPIOS) - set(le_mun.classes_)
if mun_falt_enc:
    le_mun.classes_ = np.concatenate([le_mun.classes_, np.array(sorted(mun_falt_enc))])
    print(f"  → LabelEncoder extendido con {len(mun_falt_enc)} municipios extra")

# Split temporal: train 2020-2023 / test 2024
X = df_model[FEATURES]
y_log1p = np.log1p(df_model["duracion_dias"])

mask_train = df_model["anio_radicacion"] < 2024
mask_test  = df_model["anio_radicacion"] == 2024

X_train, y_train = X[mask_train], y_log1p[mask_train]
X_test,  y_test  = X[mask_test],  y_log1p[mask_test]

print(f"Split temporal:")
print(f"  Train 2020-2023 : {len(X_train):,}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Test  2024      : {len(X_test):,}  ({len(X_test)/len(X)*100:.1f}%)")
print(f"Features          : {FEATURES}")


## Sección 5 — Comparación de modelos

Ridge, Lasso y GradientBoosting como baseline antes del XGBoost con grid-search.
Usamos `sklearn.pipeline.Pipeline` + `ColumnTransformer` para encapsular cada modelo.


In [ ]:
# Pipeline base: escalar numéricas + pasar encodings ya hechos
num_features = ["ivf_score", "carga_despacho", "n_audiencias", "apelacion",
                "anio_radicacion"]
cat_features = ["tipo_proceso_enc", "municipio_enc", "despacho_enc"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", "passthrough", cat_features),
])

MODELOS_BASELINE = {
    "Ridge":    Pipeline([("pre", preprocessor), ("model", Ridge(alpha=1.0))]),
    "Lasso":    Pipeline([("pre", preprocessor), ("model", Lasso(alpha=0.5, max_iter=5000))]),
    "GBM":      Pipeline([("pre", preprocessor), ("model", GradientBoostingRegressor(
                    n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42))]),
}

resultados_base = {}
print(f"{'Modelo':<12} {'MAPE':>7} {'MAE':>8} {'R²':>7} {'RMSE':>8}")
print("-" * 45)

for nombre, pipe in MODELOS_BASELINE.items():
    pipe.fit(X_train, y_train)
    yp_log = np.maximum(pipe.predict(X_test), 0)
    yp = np.expm1(yp_log)
    yt = np.expm1(y_test.values)
    m = {
        "MAPE": mape(yt, yp),
        "MAE":  mean_absolute_error(yt, yp),
        "R2":   r2_score(yt, yp),
        "RMSE": np.sqrt(np.mean((yt - yp)**2)),
    }
    resultados_base[nombre] = m
    print(f"{nombre:<12} {m['MAPE']:>6.1f}%  {m['MAE']:>7.1f}d  {m['R2']:>6.3f}  {m['RMSE']:>7.1f}d")


In [ ]:
# Diagnóstico overfitting — comparar train vs test en Ridge
from sklearn.metrics import mean_squared_error

pipe_ridge = MODELOS_BASELINE["Ridge"]
train_rmse = np.sqrt(mean_squared_error(
    np.expm1(y_train.values),
    np.expm1(np.maximum(pipe_ridge.predict(X_train), 0))
))
test_rmse  = np.sqrt(mean_squared_error(
    np.expm1(y_test.values),
    np.expm1(np.maximum(pipe_ridge.predict(X_test), 0))
))

print(f"Ridge — RMSE train: {train_rmse:.1f} d  |  RMSE test: {test_rmse:.1f} d")
ratio = test_rmse / train_rmse
if ratio < 1.15:
    print(f"✅ Sin sobreajuste evidente (ratio={ratio:.2f} < 1.15)")
else:
    print(f"⚠ Posible overfitting (ratio={ratio:.2f})")

# Residual plot Ridge
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (X_s, y_s, tag) in zip(axes, [(X_train, y_train,"Train"), (X_test, y_test,"Test")]):
    yp = np.expm1(np.maximum(pipe_ridge.predict(X_s), 0))
    yt = np.expm1(y_s.values)
    ax.scatter(yp, yt - yp, alpha=0.3, s=10, color="#4C72B0" if tag=="Train" else "#C44E52")
    ax.axhline(0, color="gray", linestyle="--")
    ax.set_title(f"Residuos Ridge — {tag}")
    ax.set_xlabel("Predicho (días)"); ax.set_ylabel("Residuo (días)")
plt.tight_layout(); plt.show()


## Sección 6 — Entrenamiento final XGBoost con grid-search

In [ ]:
assert XGB_OK, "Instalar XGBoost: pip install xgboost"

param_grid = {
    "n_estimators":    [300, 500],
    "learning_rate":   [0.02, 0.04, 0.06],
    "max_depth":       [4, 6],
    "subsample":       [0.70, 0.85],
    "colsample_bytree":[0.75, 0.85],
}

best_mape_val = float("inf")
best_model    = None
best_params   = None
search_results = []

print(f"Grid search: {len(list(ParameterGrid(param_grid)))} combinaciones...")

for params in ParameterGrid(param_grid):
    m = xgb.XGBRegressor(
        **params, random_state=42, verbosity=0, n_jobs=-1,
        reg_alpha=0.3, reg_lambda=1.0,
    )
    m.fit(X_train, y_train)
    yp = np.expm1(np.maximum(m.predict(X_test), 0))
    yt = np.expm1(y_test.values)
    mp = mape(yt, yp)
    search_results.append({"params": params, "mape": mp})
    if mp < best_mape_val:
        best_mape_val = mp
        best_model    = m
        best_params   = params

model = best_model
search_results.sort(key=lambda r: r["mape"])

print(f"
Top 5 configuraciones:")
print(f"{'MAPE':>7}  Params")
for r in search_results[:5]:
    print(f"  {r['mape']:>5.2f}%  {r['params']}")

print(f"
✅ Mejor modelo: MAPE={best_mape_val:.2f}%  params={best_params}")


## Sección 7 — Evaluación completa y diagnóstico de overfitting

In [ ]:
y_pred_log = np.maximum(model.predict(X_test), 0)
y_pred     = np.expm1(y_pred_log)
y_true     = np.expm1(y_test.values)

y_train_pred = np.expm1(np.maximum(model.predict(X_train), 0))
y_train_true = np.expm1(y_train.values)

metrics_test  = {"MAPE": mape(y_true, y_pred),
                 "MAE":  mean_absolute_error(y_true, y_pred),
                 "R2":   r2_score(y_true, y_pred),
                 "RMSE": np.sqrt(np.mean((y_true - y_pred)**2))}
metrics_train = {"MAPE": mape(y_train_true, y_train_pred),
                 "MAE":  mean_absolute_error(y_train_true, y_train_pred),
                 "R2":   r2_score(y_train_true, y_train_pred),
                 "RMSE": np.sqrt(np.mean((y_train_true - y_train_pred)**2))}

print("─" * 45)
print(f"{'Métrica':<10} {'Train':>10} {'Test':>10}")
print("─" * 45)
for k in ["MAPE","MAE","R2","RMSE"]:
    unit = "%" if k == "MAPE" else ("d" if k in ["MAE","RMSE"] else "")
    print(f"{k:<10} {metrics_train[k]:>9.2f}{unit}  {metrics_test[k]:>9.2f}{unit}")
print("─" * 45)

ratio_rmse = metrics_test["RMSE"] / metrics_train["RMSE"]
ovfit = "⚠ Posible overfitting" if ratio_rmse > 1.15 else "✅ Sin sobreajuste"
print(f"RMSE ratio (test/train): {ratio_rmse:.2f}  →  {ovfit}")
print(f"Objetivo MAPE ≤ 15%     : {'✅ CUMPLE' if metrics_test['MAPE'] <= 15 else '⚠ NO cumple'}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Predicho vs real
axes[0].scatter(y_true, y_pred, alpha=0.25, s=10, color="#4C72B0")
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--", linewidth=1.2)
axes[0].set_title("Predicho vs Real — Test 2024")
axes[0].set_xlabel("Real (días)"); axes[0].set_ylabel("Predicho (días)")

# 2. Residuos train vs test (KDE)
res_train = y_train_true - y_train_pred
res_test  = y_true - y_pred
axes[1].hist(res_train, bins=50, alpha=0.5, label="Train", color="#4C72B0", density=True)
axes[1].hist(res_test,  bins=50, alpha=0.5, label="Test",  color="#C44E52", density=True)
axes[1].axvline(0, color="k", linestyle="--")
axes[1].set_title("Distribución de residuos")
axes[1].set_xlabel("Residuo (días)"); axes[1].legend()

# 3. Error relativo por decil de duración real
df_eval = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
df_eval["decil"] = pd.qcut(df_eval["y_true"], 10, labels=False)
df_eval["err_abs_pct"] = np.abs(df_eval["y_true"] - df_eval["y_pred"]) / df_eval["y_true"] * 100
df_eval.groupby("decil")["err_abs_pct"].median().plot(kind="bar", ax=axes[2], color="#55A868")
axes[2].axhline(15, color="red", linestyle="--", label="Objetivo 15%")
axes[2].set_title("MAPE mediano por decil de duración")
axes[2].set_xlabel("Decil"); axes[2].set_ylabel("MAPE mediano (%)")
axes[2].legend()

plt.tight_layout(); plt.show()


## Sección 8 — Importancia de variables + SHAP

In [ ]:
FEATURE_LABELS = {
    "ivf_score":        "IVF Score (vulnerabilidad)",
    "carga_despacho":   "Carga del despacho",
    "n_audiencias":     "Número de audiencias",
    "apelacion":        "Con apelación",
    "anio_radicacion":  "Año de radicación",
    "tipo_proceso_enc": "Tipo de proceso",
    "municipio_enc":    "Municipio",
    "despacho_enc":     "Despacho asignado",
}

if hasattr(model, "feature_importances_"):
    imp = model.feature_importances_
else:
    pi = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
    imp = pi.importances_mean

imp_pct = imp / imp.sum() * 100
df_imp = pd.DataFrame({
    "feature": FEATURES,
    "label":   [FEATURE_LABELS.get(f, f) for f in FEATURES],
    "importance_pct": imp_pct.round(2),
}).sort_values("importance_pct", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(df_imp["label"], df_imp["importance_pct"], color="#4C72B0")
ax.bar_label(bars, fmt="%.1f%%", padding=4)
ax.set_title("Importancia de variables — XGBoost")
ax.set_xlabel("Importancia (%)")
plt.tight_layout(); plt.show()

df_imp.sort_values("importance_pct", ascending=False).to_csv(
    os.path.join(MODEL_DIR, "feature_importance.csv"), index=False)
print("💾 feature_importance.csv guardado")


In [ ]:
if SHAP_OK:
    explainer  = shap.TreeExplainer(model)
    X_sample   = X_test.sample(min(500, len(X_test)), random_state=42)
    shap_vals  = explainer.shap_values(X_sample)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Summary plot (beeswarm)
    plt.sca(axes[0])
    shap.summary_plot(shap_vals, X_sample, feature_names=FEATURES, show=False)
    axes[0].set_title("SHAP — Impacto por variable")

    # Bar plot de importancia media
    plt.sca(axes[1])
    shap.summary_plot(shap_vals, X_sample, feature_names=FEATURES,
                      plot_type="bar", show=False)
    axes[1].set_title("SHAP — Importancia media absoluta")

    plt.tight_layout(); plt.show()
    print("✅ SHAP calculado")
else:
    print("⚠ SHAP no disponible. Instalar con: pip install shap")
    print("   Se usó importancia nativa de XGBoost (sección anterior).")


## Sección 9 — Survival Analysis: Kaplan-Meier + Cox PH

In [ ]:
if not LIFELINES_OK:
    print("⚠ lifelines no disponible — instalar con: pip install lifelines")
else:
    # ── 9.1 Kaplan-Meier por tipo de proceso ─────────────────────────────────
    kmf = KaplanMeierFitter()
    fig, ax = plt.subplots(figsize=(10, 5))
    palette = {"ALIMENTOS":"#4C72B0","VIF":"#C44E52","HURTO_PATRIMONIAL":"#55A868","SUSTANCIAS":"#DD8452"}

    for tipo, grp in df_exp.groupby("tipo_proceso"):
        kmf.fit(grp["duracion_dias"], event_observed=grp["terminado"], label=tipo)
        kmf.plot_survival_function(ax=ax, ci_show=True, color=palette.get(tipo))

    ax.set_title("Kaplan-Meier — Probabilidad de proceso activo por tipo")
    ax.set_xlabel("Días desde radicación"); ax.set_ylabel("P(proceso activo)")
    ax.axhline(0.5, color="gray", linestyle=":", linewidth=0.8)
    plt.legend(loc="upper right"); plt.tight_layout(); plt.show()
    print("✅ Kaplan-Meier generado")


In [ ]:
if LIFELINES_OK:
    # ── 9.2 Log-rank test ALIMENTOS vs VIF ───────────────────────────────────
    g_alim = df_exp[df_exp["tipo_proceso"] == "ALIMENTOS"]
    g_vif  = df_exp[df_exp["tipo_proceso"] == "VIF"]
    lr = logrank_test(
        g_alim["duracion_dias"], g_vif["duracion_dias"],
        event_observed_A=g_alim["terminado"], event_observed_B=g_vif["terminado"]
    )
    print(f"Log-rank test ALIMENTOS vs VIF: p-value = {lr.p_value:.4f}")
    print("  Diferencia estadísticamente significativa" if lr.p_value < 0.05 else "  Sin diferencia significativa")


In [ ]:
if LIFELINES_OK:
    # ── 9.3 Cox Proportional Hazards ──────────────────────────────────────────
    # Preparar dataset con variables numéricas y codificadas
    df_cox = df_exp[["duracion_dias","terminado","ivf_score","carga_despacho",
                     "n_audiencias","apelacion","anio_radicacion"]].copy()
    df_cox = pd.get_dummies(
        df_cox.join(df_exp["tipo_proceso"]),
        columns=["tipo_proceso"], drop_first=True
    )

    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(df_cox, duration_col="duracion_dias", event_col="terminado")

    print(cph.summary[["coef","exp(coef)","p"]].sort_values("coef").to_string())

    fig, ax = plt.subplots(figsize=(8, 5))
    cph.plot(ax=ax)
    ax.set_title("Cox PH — Hazard Ratios  (HR > 1 = resuelve más rápido)")
    ax.axvline(0, color="red", linestyle="--", linewidth=1)
    plt.tight_layout(); plt.show()

    joblib.dump(cph, os.path.join(MODEL_DIR, "modelo_cox.pkl"))
    print("💾 modelo_cox.pkl guardado")


## Sección 10 — Alertas tempranas

In [ ]:
# Percentiles históricos P75 por tipo × despacho
percentiles = (
    df_exp
    .groupby(["tipo_proceso","despacho"])["duracion_dias"]
    .quantile(0.75).reset_index()
    .rename(columns={"duracion_dias":"p75_duracion"})
)

# Predicciones sobre todos los expedientes
df_alerta = df_model.copy()
preds_log  = np.maximum(model.predict(X), 0)
df_alerta["duracion_estimada"] = np.expm1(preds_log).round().astype(int)
df_alerta = df_alerta.merge(percentiles, on=["tipo_proceso","despacho"], how="left")

df_alerta["riesgo"] = "Bajo"
df_alerta.loc[df_alerta["duracion_estimada"] > df_alerta["p75_duracion"],       "riesgo"] = "Medio"
df_alerta.loc[df_alerta["duracion_estimada"] > df_alerta["p75_duracion"] * 1.5, "riesgo"] = "Alto"

print("── DISTRIBUCIÓN DE ALERTAS ──")
print(df_alerta["riesgo"].value_counts().to_string())
print()

# Exportar alertas activas (Medio + Alto)
df_alertas_activas = (
    df_alerta[df_alerta["riesgo"] != "Bajo"]
    .sort_values(["riesgo","duracion_estimada"], ascending=[True, False])
)
df_alertas_activas.to_csv("lexdata_alertas_tempranas.csv", index=False, encoding="utf-8-sig")
print("💾 lexdata_alertas_tempranas.csv guardado")


In [ ]:
# Visualizar distribución de alertas por municipio (top 20)
alertas_mun = (
    df_alerta[df_alerta["riesgo"] != "Bajo"]
    .groupby(["municipio","riesgo"])
    .size().reset_index(name="casos")
)
top20_mun = alertas_mun.groupby("municipio")["casos"].sum().nlargest(20).index
pivot_alerta = (
    alertas_mun[alertas_mun["municipio"].isin(top20_mun)]
    .pivot_table(index="municipio", columns="riesgo", values="casos", fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 7))
colors_alerta = {"Alto":"#C44E52","Medio":"#DD8452"}
bottom = np.zeros(len(pivot_alerta))
for nivel in ["Medio","Alto"]:
    if nivel in pivot_alerta.columns:
        ax.barh(pivot_alerta.index, pivot_alerta[nivel], left=bottom,
                color=colors_alerta[nivel], label=nivel)
        bottom += pivot_alerta[nivel].values
ax.set_title("Alertas tempranas por municipio (top 20)")
ax.set_xlabel("Casos en alerta")
ax.legend(title="Riesgo")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

print(f"
Top 10 casos en riesgo ALTO:")
cols_show = ["expediente_id","municipio","tipo_proceso","despacho",
             "ivf_score","duracion_estimada","p75_duracion","riesgo"]
print(df_alertas_activas[df_alertas_activas["riesgo"]=="Alto"][cols_show].head(10).to_string(index=False))


## Sección 11 — Serialización y resumen de outputs

In [ ]:
# ── Serializar modelo XGBoost ─────────────────────────────────────────────────
modelo_data = {
    "modelo":    model,
    "nombre":    "XGBoost",
    "features":  FEATURES,
    "le_tipo":   le_tipo,
    "le_mun":    le_mun,
    "le_desp":   le_desp,
    "mape":      metrics_test["MAPE"],
    "params":    best_params,
}
modelo_path = os.path.join(MODEL_DIR, "modelo_regresion.pkl")
joblib.dump(modelo_data, modelo_path)
print(f"💾 {modelo_path}")

# ── Verificación rápida de carga ──────────────────────────────────────────────
loaded   = joblib.load(modelo_path)
le_check = loaded["le_mun"]
missing  = set(ALL_MUNICIPIOS) - set(le_check.classes_)
print(f"✅ Modelo cargado — {len(le_check.classes_)} municipios en encoder")
print(f"   Municipios faltantes en encoder: {len(missing)}")
print(f"   Cobertura: {'COMPLETA ✅' if not missing else f'INCOMPLETA ⚠ {missing}'}")


In [ ]:
# ── Resumen final ─────────────────────────────────────────────────────────────
sep = "=" * 62
print(sep)
print("OUTPUTS — LexData Modelo Predictivo v1")
print(sep)

outputs = [
    (modelo_path,                                "modelo_regresion.pkl"),
    (os.path.join(MODEL_DIR,"feature_importance.csv"), "feature_importance.csv"),
    (os.path.join(MODEL_DIR,"modelo_cox.pkl"),   "modelo_cox.pkl (survival)"),
    ("lexdata_alertas_tempranas.csv",             "lexdata_alertas_tempranas.csv"),
]
for ruta, desc in outputs:
    existe = os.path.exists(ruta)
    tam    = f"({os.path.getsize(ruta)/1024:.0f} KB)" if existe else ""
    icono  = "✅" if existe else "⚠ no generado"
    print(f"  {icono}  {desc:<40} {tam}")

print()
print("MÉTRICAS (XGBoost — Test 2024):")
print(f"  MAPE : {metrics_test['MAPE']:>6.1f}%   (objetivo ≤ 15%  {'✅' if metrics_test['MAPE']<=15 else '⚠'})")
print(f"  MAE  : {metrics_test['MAE']:>6.0f} días")
print(f"  R²   : {metrics_test['R2']:>6.3f}")
print(f"  RMSE : {metrics_test['RMSE']:>6.0f} días")
print()
print("SIGUIENTE PASO → streamlit run streamlit/app.py")
print(sep)
